# Pruebas de Embeddings - Español/Shiwilu

Este notebook permite probar interactivamente los modelos de embeddings:
- **FastText**: Similitud morfológica (subpalabras)
- **Sentence Transformers**: Similitud semántica cross-lingual

In [1]:
# Imports
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Agregar el directorio raíz al path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: C:\Users\Personal\Desktop\Recopilatorio\Tesis\Desarrollo


## 1. Cargar Modelos

In [2]:
# Cargar FastText
from gensim.models import FastText

FASTTEXT_PATH = PROJECT_ROOT / "models" / "fasttext" / "fasttext.model"
ft_model = FastText.load(str(FASTTEXT_PATH))

print(f"FastText cargado: {len(ft_model.wv.key_to_index):,} palabras en vocabulario")

FastText cargado: 7,480 palabras en vocabulario


In [3]:
# Cargar Sentence Transformers
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim

st_model = SentenceTransformer("intfloat/multilingual-e5-small")

print("Sentence Transformers cargado: intfloat/multilingual-e5-small")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sentence Transformers cargado: intfloat/multilingual-e5-small


In [4]:
# Cargar corpus para referencia
CORPUS_PATH = PROJECT_ROOT / "data" / "processed" / "03_pre_embeddings" / "dataset_pre_embeddings.csv"
df = pd.read_csv(CORPUS_PATH, encoding="utf-8-sig")

print(f"Corpus cargado: {len(df):,} pares")
df.head()

Corpus cargado: 3,207 pares


,pair_id,ESP_original,SHIWILU_original,ESP_normalizado,SHIWILU_normalizado,source,source_pair_id,has_audit_flags
0,U00000,Ve.,PAKER',ve.,paker',flashcards,P00000,True
1,U00001,Vete.,PAKER',vete.,paker',flashcards,P00001,True
2,U00002,Vaya.,PAKER',vaya.,paker',flashcards,P00002,True
3,U00003,Hola.,MA'PU'SIN,hola.,ma'pu'sin,flashcards,P00003,False
4,U00004,¡Corre!,TEKKER',¡corre!,tekker',flashcards,P00004,False


## 2. Funciones de Prueba

In [6]:
def fasttext_similar(word: str, topn: int = 10):
    """
    Encuentra palabras similares usando FastText.
    Funciona para palabras OOV gracias a subpalabras.
    """
    try:
        results = ft_model.wv.most_similar(word, topn=topn)
        print(f"\n🔤 FastText - Palabras similares a '{word}':")
        print("-" * 50)
        for w, score in results:
            print(f"  {w:<30} {score:.4f}")
        return results
    except Exception as e:
        print(f"Error: {e}")
        return []

In [7]:
def sentence_similarity(text1: str, text2: str):
    """
    Calcula similitud entre dos textos usando Sentence Transformers.
    Útil para comparar oraciones español-shiwilu.
    """
    emb1 = st_model.encode(f"query: {text1}", convert_to_numpy=True)
    emb2 = st_model.encode(f"query: {text2}", convert_to_numpy=True)
    
    similarity = cos_sim(emb1, emb2).item()
    
    print(f"\n🌐 Sentence Transformers - Similitud:")
    print("-" * 50)
    print(f"  Texto 1: {text1}")
    print(f"  Texto 2: {text2}")
    print(f"  Score:   {similarity:.4f}")
    
    return similarity

In [8]:
def buscar_en_corpus(texto: str, columna: str = "ESP_normalizado", limit: int = 5):
    """
    Busca un texto en el corpus y muestra los pares correspondientes.
    columna: 'ESP_normalizado' o 'SHIWILU_normalizado'
    """
    matches = df[df[columna].str.contains(texto, case=False, na=False)]
    
    print(f"\n📚 Búsqueda en corpus - '{texto}' en {columna}:")
    print("-" * 50)
    
    if len(matches) == 0:
        print("  No se encontraron coincidencias")
        return None
    
    for _, row in matches.head(limit).iterrows():
        print(f"  ESP: {row['ESP_normalizado']}")
        print(f"  SHI: {row['SHIWILU_normalizado']}")
        print()
    
    return matches.head(limit)

In [9]:
def comparar_par(esp: str, shi: str):
    """
    Compara un par español-shiwilu con ambos métodos.
    """
    print(f"\n{'='*60}")
    print(f"COMPARACIÓN DE PAR")
    print(f"{'='*60}")
    print(f"ESP:     {esp}")
    print(f"SHIWILU: {shi}")
    print()
    
    # Sentence Transformers (oraciones completas)
    st_score = sentence_similarity(esp, shi)
    
    # FastText (palabra por palabra)
    esp_words = esp.split()[:3]
    shi_words = shi.split()[:3]
    
    print(f"\n🔤 FastText - Similitudes palabra a palabra:")
    print("-" * 50)
    
    for ew in esp_words:
        for sw in shi_words:
            try:
                sim = ft_model.wv.similarity(ew, sw)
                print(f"  {ew} <-> {sw}: {sim:.4f}")
            except:
                print(f"  {ew} <-> {sw}: [error]")
    
    return st_score

## 3. Pruebas Interactivas

Modifica las celdas de abajo para probar con tus propias palabras.

In [10]:
# ========================================
# PRUEBA 1: Palabras similares (FastText)
# ========================================

# Cambia esta palabra por la que quieras probar
PALABRA = "hola"

fasttext_similar(PALABRA, topn=10)


🔤 FastText - Palabras similares a 'hola':
--------------------------------------------------
  bulla                          0.9704
  ábrela                         0.9687
  doncella                       0.9621
  baila                          0.9609
  pásenla                        0.9557
  olla                           0.9542
  pásala                         0.9486
  suéltala                       0.9396
  fila                           0.9324
  icáramela                      0.9323


[('bulla', 0.9703963398933411),
 ('ábrela', 0.96868896484375),
 ('doncella', 0.9621127843856812),
 ('baila', 0.9608981013298035),
 ('pásenla', 0.955747663974762),
 ('olla', 0.9541836380958557),
 ('pásala', 0.9485902786254883),
 ('suéltala', 0.9395923018455505),
 ('fila', 0.9324290752410889),
 ('icáramela', 0.9323363900184631)]

In [11]:
# ========================================
# PRUEBA 2: Palabra en Shiwilu
# ========================================

PALABRA_SHIWILU = "paker'"

fasttext_similar(PALABRA_SHIWILU, topn=10)


🔤 FastText - Palabras similares a 'paker'':
--------------------------------------------------
  tuker'                         0.9945
  li'ker'                        0.9931
  du'ker'                        0.9930
  da'ker'                        0.9923
  wer'ker'                       0.9921
  tu'neker'                      0.9917
  lia'ker'                       0.9912
  anu'ker'                       0.9910
  duker'                         0.9907
  i'shi'ker'                     0.9901


[("tuker'", 0.9944687485694885),
 ("li'ker'", 0.9930895566940308),
 ("du'ker'", 0.9929817914962769),
 ("da'ker'", 0.992343008518219),
 ("wer'ker'", 0.9920505881309509),
 ("tu'neker'", 0.9916883707046509),
 ("lia'ker'", 0.9912275671958923),
 ("anu'ker'", 0.9910310506820679),
 ("duker'", 0.9907094240188599),
 ("i'shi'ker'", 0.990077793598175)]

In [12]:
# ========================================
# PRUEBA 3: Similitud entre oraciones
# ========================================

TEXTO_ESP = "hola"
TEXTO_SHI = "ma'pu'sin"

sentence_similarity(TEXTO_ESP, TEXTO_SHI)


🌐 Sentence Transformers - Similitud:
--------------------------------------------------
  Texto 1: hola
  Texto 2: ma'pu'sin
  Score:   0.8075


0.8074854612350464

In [13]:
# ========================================
# PRUEBA 4: Buscar en el corpus
# ========================================

BUSCAR = "agua"

buscar_en_corpus(BUSCAR, columna="ESP_normalizado")


📚 Búsqueda en corpus - 'agua' en ESP_normalizado:
--------------------------------------------------
  ESP: aguarda aquí.
  SHI: wa'tenter' asek

  ESP: ¡oye, aguarda!
  SHI: ¡lau'ker', wa'tenter'!

  ESP: necesito agua.
  SHI: luwantulek dek'

  ESP: aguanta.
  SHI: yan'katekter'

  ESP: entonces cuando se termina, nuevamente la yuca sobrante se hierve (con el agua nueva que ha puesto) y se vuelve a trasladar.
  SHI: ipa’linchi ta’wantununta’sik nu’nipi’la ñapalli ker‘ ukluka’sik upetñunta’lek.



,pair_id,ESP_original,SHIWILU_original,ESP_normalizado,SHIWILU_normalizado,source,source_pair_id,has_audit_flags
444,U00444,Aguarda aquí.,WA'TENTER' ASEK,aguarda aquí.,wa'tenter' asek,flashcards,P00444,True
1052,U01052,"¡Oye, aguarda!","¡LAU'KER', WA'TENTER'!","¡oye, aguarda!","¡lau'ker', wa'tenter'!",flashcards,P01052,False
1077,U01077,Necesito agua.,LUWANTULEK DEK',necesito agua.,luwantulek dek',flashcards,P01077,False
1358,U01358,Aguanta.,YAN'KATEKTER',aguanta.,yan'katekter',flashcards,P01358,False
2369,U02369,"Entonces cuando se termina, nuevamente la yuca...",Ipa’linchi ta’wantununta’sik nu’nipi’la ñapall...,"entonces cuando se termina, nuevamente la yuca...",ipa’linchi ta’wantununta’sik nu’nipi’la ñapall...,pdf_textos,T002_N036,True


In [14]:
# ========================================
# PRUEBA 5: Comparación completa de un par
# ========================================

ESP = "buenos días"
SHI = "ma'pu' ekkennala"

comparar_par(ESP, SHI)


COMPARACIÓN DE PAR
ESP:     buenos días
SHIWILU: ma'pu' ekkennala


🌐 Sentence Transformers - Similitud:
--------------------------------------------------
  Texto 1: buenos días
  Texto 2: ma'pu' ekkennala
  Score:   0.8250

🔤 FastText - Similitudes palabra a palabra:
--------------------------------------------------
  buenos <-> ma'pu': 0.3353
  buenos <-> ekkennala: 0.3307
  días <-> ma'pu': 0.3693
  días <-> ekkennala: 0.3145


0.8250374794006348

## 4. Exploración del Vocabulario

In [ ]:
# Ver algunas palabras del vocabulario FastText
vocab = list(ft_model.wv.key_to_index.keys())

print("Palabras en español (muestra):")
esp_words = [w for w in vocab if not "'" in w][:20]
print(esp_words)

print("\nPalabras en shiwilu (muestra):")
shi_words = [w for w in vocab if "'" in w][:20]
print(shi_words)

In [ ]:
# Estadísticas del vocabulario
total = len(vocab)
with_apostrophe = len([w for w in vocab if "'" in w])
without_apostrophe = total - with_apostrophe

print(f"Total vocabulario: {total:,}")
print(f"Sin apóstrofe (mayormente español): {without_apostrophe:,}")
print(f"Con apóstrofe (mayormente shiwilu): {with_apostrophe:,}")